In [0]:
from pyspark.sql.functions import input_file_name
import requests
import json
from pyspark.sql.types import StructType, StructField, StringType

# 1. Read validated control table
control_df = spark.read.table("control.pipeline_config_batch")

# 2. API Ingestion Function
def ingest_api(source):
    url = source['base_url']
    params = source.get('params', {})
    bronze_path = source['bronze_path']
    
    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()
    
    entries = data.get("entry", [])
    resources = [entry["resource"] for entry in entries if "resource" in entry]
    
    flattened = []
    for res in resources:
        obs = {
            "id": res.get("id"),
            "status": res.get("status"),
            "category": str(res.get("category")),
            "code": str(res.get("code")),
            "subject": str(res.get("subject")),
            "effectiveDateTime": res.get("effectiveDateTime")
        }
        flattened.append(obs)

    schema = StructType([
        StructField("id", StringType(), True),
        StructField("status", StringType(), True),
        StructField("category", StringType(), True),
        StructField("code", StringType(), True),
        StructField("subject", StringType(), True),
        StructField("effectiveDateTime", StringType(), True)
    ])
    
    if flattened:
        df = spark.createDataFrame(flattened, schema)
        df.write.mode("append").json(bronze_path)
        print(f"✅ Ingested {len(flattened)} records from API to: {bronze_path}")
    else:
        print("⚠️ No data returned from API")

# 3. PDF Ingestion Function
def ingest_pdfs(source):
    bronze_path = source['bronze_path']
    base_path = source['base_path']
    file_pattern = source.get('file_pattern', '*.pdf')

    df = (spark.read.format("binaryFile")
                .option("pathGlobFilter", file_pattern)
                .load(base_path))
    
    df = df.withColumn("source_path", input_file_name())
    df.write.mode("append").json(bronze_path)
    print(f"✅ Ingested PDF metadata to: {bronze_path}")

# 4. Execute ingestion for each row in control table
for row in control_df.collect():
    src = row.asDict()
    if src['type'] == 'api':
        ingest_api(src)
    elif src['type'] == 'file':
        ingest_pdfs(src)



✅ Ingested 100 records from API to: /FileStore/bronze/bronze_healthcare_patients
✅ Ingested PDF metadata to: /FileStore/bronze/bronze_pdfs


In [0]:
display(dbutils.fs.ls("/FileStore/bronze/bronze_healthcare_patients"))


path,name,size,modificationTime
dbfs:/FileStore/bronze/bronze_healthcare_patients/_SUCCESS,_SUCCESS,0,1746011622000
dbfs:/FileStore/bronze/bronze_healthcare_patients/_committed_1605043753016218440,_committed_1605043753016218440,201,1745964414000
dbfs:/FileStore/bronze/bronze_healthcare_patients/_committed_3570187116925635366,_committed_3570187116925635366,380,1745965161000
dbfs:/FileStore/bronze/bronze_healthcare_patients/_committed_393455864987289604,_committed_393455864987289604,376,1745965197000
dbfs:/FileStore/bronze/bronze_healthcare_patients/_committed_540912048085536586,_committed_540912048085536586,376,1745965140000
dbfs:/FileStore/bronze/bronze_healthcare_patients/_committed_6233962395319180748,_committed_6233962395319180748,380,1746011622000
dbfs:/FileStore/bronze/bronze_healthcare_patients/_committed_8122111417362697828,_committed_8122111417362697828,380,1745965080000
dbfs:/FileStore/bronze/bronze_healthcare_patients/_committed_vacuum7426382881171016416,_committed_vacuum7426382881171016416,226,1746011622000
dbfs:/FileStore/bronze/bronze_healthcare_patients/_started_6233962395319180748,_started_6233962395319180748,0,1746011621000
dbfs:/FileStore/bronze/bronze_healthcare_patients/part-00000-tid-1605043753016218440-f88e6a02-23be-4f70-99b6-e72d7b2cdcbd-7-1-c000.json,part-00000-tid-1605043753016218440-f88e6a02-23be-4f70-99b6-e72d7b2cdcbd-7-1-c000.json,0,1745964413000


In [0]:
bronze_healthcare_df = spark.read.json("/FileStore/bronze/bronze_healthcare_patients")
display(bronze_healthcare_df)



birthDate,first_name,gender,id,last_name
null,XYZ,male,597209,ABC
1985-09-09,Gris,female,597211,Meth
2020-02-02T22:00:00.000Z,XYZ,male,597210,ABC
1985-09-09,Gris,female,597212,Meth
1985-09-09,Grishma,female,597213,Methaila
1994-11-17,Sowmya,female,597173,Mellatur Sreedhar
1995-08-16,Ed,male,597217,Tan
null,Greg,null,597219,Juniper
null,Dillon,male,597220,Thompson
1910-04-01,Andrea,female,597224,Frazier
